<a href="https://colab.research.google.com/github/mohammedh897/Prediction-of-Product-Sales/blob/main/Project_1_Part_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# load data
import pandas as pd
path='/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week05/Data/sales_predictions_2023 (1).csv'
df = pd.read_csv(path)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


### Part 5: Sales Prediction Project - Data Loading and Preprocessing

Reloading the original dataset to ensure no data leakage and starting the cleaning process.

#### Step 1: Handle Duplicates and Inconsistent Categorical Data

### Handle Duplicates

In [4]:
duplicates = df.duplicated().sum()
duplicates

np.int64(0)

### Handle Inconsistent Categorical Data


In [5]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object')

In [6]:
df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
Low Fat,5089
Regular,2889
LF,316
reg,117
low fat,112


In [7]:
#Fix inconsistent categories in Item_Fat_Content
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'low fat',
    'Low Fat': 'low fat',
    'low fat': 'low fat',
    'Reg': 'regular',
    'reg': 'regular',
    'Regular': 'regular'
})


In [8]:
# Check value counts in Item_Fat_Content after cleaning

df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
low fat,5517
regular,3006


#### Step 2: Identify Features (X) and Target (y), Drop 'Item_Identifier'

In [9]:
# Define target variable
y = df['Item_Outlet_Sales']

# Define features, dropping 'Item_Identifier' and the target variable
X = df.drop(columns=['Item_Identifier', 'Item_Outlet_Sales'])


#### Step 3: Perform Train-Test Split

In [10]:
from sklearn.model_selection import train_test_split
# Perform 75/25 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

Identify each feature as numerical, ordinal, or nominal.

In [11]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6392 entries, 4776 to 7270
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Weight                5285 non-null   float64
 1   Item_Fat_Content           6392 non-null   object 
 2   Item_Visibility            6392 non-null   float64
 3   Item_Type                  6392 non-null   object 
 4   Item_MRP                   6392 non-null   float64
 5   Outlet_Identifier          6392 non-null   object 
 6   Outlet_Establishment_Year  6392 non-null   int64  
 7   Outlet_Size                4580 non-null   object 
 8   Outlet_Location_Type       6392 non-null   object 
 9   Outlet_Type                6392 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 549.3+ KB


In [12]:
# Checking object columns
X_train.select_dtypes('object')

,Item_Fat_Content,Item_Type,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type
4776,low fat,Household,OUT018,Medium,Tier 3,Supermarket Type2
7510,regular,Snack Foods,OUT018,Medium,Tier 3,Supermarket Type2
5828,regular,Meat,OUT049,Medium,Tier 1,Supermarket Type1
5327,low fat,Baking Goods,OUT035,Small,Tier 2,Supermarket Type1
4810,low fat,Frozen Foods,OUT045,NaN,Tier 2,Supermarket Type1
...,...,...,...,...,...,...
5734,regular,Fruits and Vegetables,OUT010,NaN,Tier 3,Grocery Store
5191,low fat,Frozen Foods,OUT017,NaN,Tier 2,Supermarket Type1
5390,low fat,Health and Hygiene,OUT045,NaN,Tier 2,Supermarket Type1
860,low fat,Snack Foods,OUT017,NaN,Tier 2,Supermarket Type1


In [13]:
X_train['Item_Fat_Content'].value_counts(dropna=False)

,count
Item_Fat_Content,
low fat,4129
regular,2263


In [14]:
X_train['Item_Type'].value_counts(dropna=False)

,count
Item_Type,
Fruits and Vegetables,948
Snack Foods,906
Household,695
Frozen Foods,632
Dairy,507
Canned,481
Baking Goods,478
Health and Hygiene,390
Soft Drinks,331


In [15]:
X_train['Outlet_Identifier'].value_counts(dropna=False)

,count
Outlet_Identifier,
OUT027,723
OUT035,709
OUT018,704
OUT045,699
OUT017,698
OUT046,695
OUT013,689
OUT049,676
OUT010,415


In [16]:
X_train['Outlet_Size'].value_counts(dropna=False)

,count
Outlet_Size,
Medium,2103
NaN,1812
Small,1788
High,689


In [17]:
X_train['Outlet_Location_Type'].value_counts(dropna=False)

,count
Outlet_Location_Type,
Tier 3,2531
Tier 2,2106
Tier 1,1755


In [18]:
X_train['Outlet_Type'].value_counts(dropna=False)

,count
Outlet_Type,
Supermarket Type1,4166
Grocery Store,799
Supermarket Type3,723
Supermarket Type2,704


## Features:

- Ordinal: Outlet_Size, Outlet_Location_Type
- Categorical: Item_Fat_Content, Item_Type,  Outlet_Identifier, Outlet_Type.
- The remaining features are numeric.
Create 3 pipelines (one for numeric, ordinal, and categorical features).


#### Step 4: Create Preprocessing Object (ColumnTransformer)

Imputation will be part of the preprocessing pipeline and will be applied *after* the train-test split.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

# Identify numerical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Define ordinal features and their explicit orderings
ordinal_features = ['Outlet_Size', 'Outlet_Location_Type']
outlet_size_order = ['Small', 'Medium', 'High']
outlet_location_type_order = ['Tier 1', 'Tier 2', 'Tier 3']

# Identify nominal features (remaining object columns after excluding ordinal ones)
all_object_features = X.select_dtypes(include=['object']).columns.tolist()
nominal_features = [col for col in all_object_features if col not in ordinal_features]

# Define preprocessing steps for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Impute missing numerical values with the mean
    ('scaler', StandardScaler()) # Scale numerical features
])

# Define preprocessing steps for ordinal features
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing ordinal values with the most frequent
    ('ordinal', OrdinalEncoder(categories=[outlet_size_order, outlet_location_type_order], handle_unknown='use_encoded_value', unknown_value=-1)), # Ordinal encode with specified order
    ('scaler', StandardScaler()) # Scale ordinal features after encoding
])

# Define preprocessing steps for nominal features
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing nominal values with the most frequent
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # One-hot encode nominal features, ensuring dense output
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features)
    ])

In [20]:
preprocessor.fit(X_train)
# Transform the training and test data
X_train_tf = preprocessor.transform(X_train)
X_test_tf = preprocessor.transform(X_test)

# Get feature names after preprocessing
numerical_feature_names = numerical_features
ordinal_feature_names = ordinal_features

# Get nominal feature names from the OneHotEncoder that is part of the 'nom' transformer
nominal_transformer_pipeline = preprocessor.named_transformers_['nom']
onehot_encoder = nominal_transformer_pipeline.named_steps['onehot']
nominal_feature_names = onehot_encoder.get_feature_names_out(nominal_features)

# Combine all feature names in the correct order
all_feature_names = numerical_feature_names + ordinal_feature_names + nominal_feature_names.tolist()

# Convert the processed arrays back to DataFrames
X_train_tf = pd.DataFrame(X_train_tf, columns=all_feature_names, index=X_train.index)
X_test_tf = pd.DataFrame(X_test_tf, columns=all_feature_names, index=X_test.index)

# Display the first few rows of the processed training DataFrame
display(X_train_tf.head())

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Fat_Content_low fat,Item_Fat_Content_regular,Item_Type_Baking Goods,Item_Type_Breads,...,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
4776,0.817249,-0.712775,1.828109,1.327849,0.287374,1.084948,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7510,0.556340,-1.291052,0.603369,1.327849,0.287374,1.084948,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5828,-0.131512,1.813319,0.244541,0.136187,0.287374,-1.384777,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
5327,-1.169219,-1.004931,-0.952591,0.732018,-1.384048,-0.149914,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4810,1.528819,-0.965484,-0.336460,0.493686,0.287374,-0.149914,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [21]:
def regression_metrics(y_true, y_pred, label='', verbose=True, output_dict=False):
    mae       = mean_absolute_error(y_true, y_pred)
    mse       = mean_squared_error(y_true, y_pred)
    rmse      = np.sqrt(mse)
    r_squared = r2_score(y_true, y_pred)
    if verbose:
        header = "-" * 60
        print(header, f"Regression Metrics: {label}", header, sep='\n')
        print(f"- MAE  = {mae:,.3f}")
        print(f"- MSE  = {mse:,.3f}")
        print(f"- RMSE = {rmse:,.3f}")
        print(f"- R^2  = {r_squared:,.3f}")

    if output_dict:
        return {'Label': label, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R^2': r_squared}
def evaluate_regression(reg, X_train, y_train, X_test, y_test,
                        verbose=True, output_frame=False):
    y_train_pred = reg.predict(X_train)
    results_train = regression_metrics(y_train, y_train_pred, verbose=verbose,
                                       output_dict=output_frame, label='Training Data')
    print()
    y_test_pred = reg.predict(X_test)
    results_test = regression_metrics(y_test, y_test_pred, verbose=verbose,
                                      output_dict=output_frame, label='Test Data')
    if output_frame:
        results_df = pd.DataFrame([results_train, results_test])
        results_df = results_df.set_index('Label')
        results_df.index.name = None
        return results_df.round(3)

#### Step 5: Apply Preprocessing and Build Linear Regression Model

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Instantiate Linear Regression model
lin_reg = LinearRegression()

# Train the model on the preprocessed training data
lin_reg.fit(X_train_tf, y_train)

LinearRegression()

#### Evaluate Linear Regression Model

In [23]:
# Evaluate model on training and test data using the custom function
evaluate_regression(lin_reg, X_train_tf, y_train, X_test_tf, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 847.129
- MSE  = 1,297,558.136
- RMSE = 1,139.104
- R^2  = 0.562

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 804.120
- MSE  = 1,194,349.715
- RMSE = 1,092.863
- R^2  = 0.567


#### Linear Regression Model Overfit/Underfit Analysis

Comparing the R-squared values:

*   **Training Data R-squared: 0.562**
*   **Test Data R-squared: 0.567**

The R-squared values for both the training and test data are very similar, with the test R-squared being slightly higher. This indicates that the model is **not overfitting**, as it performs consistently on both seen and unseen data.

However, both R-squared values are relatively low (around 0.56). This suggests that the model is likely **underfitting**. It's not capturing a large portion of the variance in the target variable (`Item_Outlet_Sales`), meaning it might be too simple to adequately learn the underlying patterns in the data. There's significant room for improvement to capture more of the data's variability.

#### Step 6: Build and Train Random Forest Regressor Model

In [24]:
from sklearn.ensemble import RandomForestRegressor

# Instantiate Random Forest Regressor model with default parameters
rf_reg = RandomForestRegressor(random_state=42) # Set random_state for reproducibility

# Train the model on the preprocessed training data
rf_reg.fit(X_train_tf, y_train)

RandomForestRegressor(random_state=42)

#### Evaluate Random Forest Regressor Model

In [25]:
# Evaluate model on training and test data using the custom function
evaluate_regression(rf_reg, X_train_tf, y_train, X_test_tf, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 296.536
- MSE  = 182,688.635
- RMSE = 427.421
- R^2  = 0.938

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 766.767
- MSE  = 1,217,003.043
- RMSE = 1,103.179
- R^2  = 0.559


#### Random Forest Regressor Model Overfit/Underfit Analysis

Comparing the R-squared values for the Random Forest Regressor:

*   **Training Data R-squared: 0.938**
*   **Test Data R-squared: 0.559**

There is a significant drop in R-squared from the training data to the test data (from 0.938 to 0.559). This large difference clearly indicates that the Random Forest model is **overfitting** the training data. The model has learned the training data too well, including its noise, and as a result, it does not generalize effectively to new, unseen data. While the training R-squared is very high, the substantially lower test R-squared suggests that the model's complexity might be too high for the given data, leading to poor performance on new samples. To address this, hyperparameter tuning would be essential to find a better balance between bias and variance.

#### Model Performance Comparison: Random Forest vs. Linear Regression

Let's compare the test scores of the two models:

**Linear Regression Test Scores:**
*   MAE  = 804.120
*   MSE  = 1,194,349.715
*   RMSE = 1,092.863
*   R^2  = 0.567

**Random Forest Regressor Test Scores:**
*   MAE  = 766.767
*   MSE  = 1,217,003.043
*   RMSE = 1,103.179
*   R^2  = 0.559

Upon comparing the test metrics:
*   The **Linear Regression** model exhibits a slightly better **R-squared (0.567 vs 0.559)**, indicating it explains a marginally higher proportion of the variance in the test data.
*   Linear Regression also has slightly lower **MSE and RMSE**, suggesting it generally makes predictions that are closer to the actual values on average.
*   Conversely, the **Random Forest** model has a slightly lower **MAE (766.767 vs 804.120)**, meaning its average absolute errors are marginally smaller.

Overall, the **Linear Regression model currently shows slightly better generalization performance on the test set**, according to R-squared, MSE, and RMSE. While the Random Forest Regressor shows strong overfitting, its MAE is better, suggesting that it might have some predictive capability despite the overfitting. Further hyperparameter tuning of the Random Forest model could potentially improve its test performance significantly and reduce overfitting.

#### Step 7: Hyperparameter Tuning for Random Forest Regressor using GridSearchCV

In [26]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for Random Forest Regressor
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_depth': [10, 20, None],      # Maximum depth of the tree (None means nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples)
    'min_samples_leaf': [1, 2, 4]     # Minimum number of samples required to be at a leaf node
}

# Instantiate the GridSearchCV object
# Using 'neg_mean_squared_error' as scoring for regression (GridSearchCV minimizes by default, so we negate MSE)
# n_jobs=-1 uses all available processors
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42), # Use the same random_state as initial model for reproducibility
    param_grid=param_grid,
    scoring='neg_mean_squared_error', # Optimize for lowest (negative) MSE
    cv=5, # 5-fold cross-validation
    n_jobs=-1, # Use all available cores
    verbose=2 # Display progress
)

# Fit GridSearchCV to the training data
grid_search.fit(X_train_tf, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits


GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2, 4],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error', verbose=2)

#### Best Parameters and Best Score from GridSearchCV

In [27]:
# Print the best hyperparameters found
print(f"Best Hyperparameters: {grid_search.best_params_}")

# Print the best score (negative MSE, so convert to positive RMSE)
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best Cross-validation RMSE: {best_rmse:,.3f}")

Best Hyperparameters: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 300}
Best Cross-validation RMSE: 1,103.523


#### Evaluate Tuned Random Forest Regressor Model

In [28]:
# Get the best model from GridSearchCV
best_rf_reg = grid_search.best_estimator_

# Evaluate the best model on training and test data using the custom function
print("Evaluating the best Random Forest model found by GridSearchCV:")
evaluate_regression(best_rf_reg, X_train_tf, y_train, X_test_tf, y_test)

Evaluating the best Random Forest model found by GridSearchCV:
------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE  = 643.075
- MSE  = 824,432.008
- RMSE = 907.982
- R^2  = 0.721

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE  = 739.404
- MSE  = 1,130,667.619
- RMSE = 1,063.329
- R^2  = 0.590


#### Tuned vs. Default Random Forest Regressor Performance Comparison

Let's compare the test scores of the **Default Random Forest Regressor** with the **Tuned Random Forest Regressor**:

**Default Random Forest Regressor Test Scores:**
*   MAE  = 766.767
*   MSE  = 1,217,003.043
*   RMSE = 1,103.179
*   R^2  = 0.559

**Tuned Random Forest Regressor Test Scores:**
*   MAE  = 739.404
*   MSE  = 1,130,667.619
*   RMSE = 1,063.329
*   R^2  = 0.590

**Analysis of Performance Improvement:**

Comparing the test metrics, the **tuned Random Forest Regressor shows improved performance** over the default Random Forest model:

*   **R-squared (R^2):** The R-squared value increased from 0.559 to **0.590**. This indicates that the tuned model explains a higher proportion of the variance in the target variable on the test set, suggesting better generalization.
*   **Mean Absolute Error (MAE):** The MAE decreased from 766.767 to **739.404**. A lower MAE means the average absolute difference between predicted and actual values is smaller, indicating more accurate predictions.
*   **Mean Squared Error (MSE):** The MSE decreased from 1,217,003.043 to **1,130,667.619**. A lower MSE indicates that the model's predictions are closer to the true values, with fewer large errors.
*   **Root Mean Squared Error (RMSE):** The RMSE decreased from 1,103.179 to **1,063.329**. Similar to MSE, a lower RMSE implies better accuracy and smaller prediction errors.

Overall, hyperparameter tuning successfully reduced the overfitting observed in the default Random Forest model and significantly improved its predictive performance and generalization ability on unseen data, as evidenced by the better scores across all evaluated metrics.

#### Final Model Recommendation

After evaluating the Linear Regression model, the default Random Forest Regressor, and the hyperparameter-tuned Random Forest Regressor, I recommend the **Tuned Random Forest Regressor** for implementation.

Here's a summary of the test set performance for all models:

*   **Linear Regression:**
    *   MAE  = 804.120
    *   MSE  = 1,194,349.715
    *   RMSE = 1,092.863
    *   R^2  = 0.567

*   **Default Random Forest Regressor:**
    *   MAE  = 766.767
    *   MSE  = 1,217,003.043
    *   RMSE = 1,103.179
    *   R^2  = 0.559

*   **Tuned Random Forest Regressor:**
    *   MAE  = 739.404
    *   MSE  = 1,130,667.619
    *   RMSE = 1,063.329
    *   R^2  = 0.590

**Justification for Recommendation:**

The **Tuned Random Forest Regressor** clearly outperforms both the Linear Regression and the default Random Forest models on the test set across all primary evaluation metrics:

1.  **Highest R-squared (R^2 = 0.590):** This indicates that the tuned model explains the largest proportion of the variance in the target variable (`Item_Outlet_Sales`) on unseen data. While still not extremely high, it's the best among the models tested.
2.  **Lowest MAE (739.404):** The average absolute difference between the predicted and actual sales is the smallest, meaning its predictions are, on average, closest to the true values.
3.  **Lowest MSE (1,130,667.619) and RMSE (1,063.329):** These metrics penalize larger errors more heavily. The lower MSE and RMSE suggest that the tuned model has fewer significant prediction errors compared to the other models.

Furthermore, the hyperparameter tuning successfully addressed the overfitting issue observed in the default Random Forest model, leading to a much better balance between training and test performance and improved generalization. Although the R-squared values for all models are in a similar range, the consistent improvement across all metrics makes the Tuned Random Forest Regressor the most robust and accurate choice among the options explored.

#### Model Performance for Non-Technical Stakeholders: Tuned Random Forest Regressor

To help our non-technical stakeholders understand the performance of our chosen **Tuned Random Forest Regressor** model, let's break down its key metrics on unseen data (test set):

**1. R-squared (R²): How much of the sales variation can we explain?**

*   **Our Model's Test R²: 0.590 (or 59%)**

In simple terms, R-squared tells us how well our model's predictions align with the actual sales figures. An R-squared of 0.590 means that our model can **explain about 59% of the variability** in Item Outlet Sales. This is a decent starting point, indicating that our model is capturing a significant portion of the factors influencing sales, but there's still 41% of variability it's not accounting for. This suggests room for future improvements in data or modeling.

**2. Mean Absolute Error (MAE): On average, how far off are our predictions?**

*   **Our Model's Test MAE: 739.404**

I've chosen **Mean Absolute Error (MAE)** to explain to stakeholders because it's the most intuitive and easy-to-understand metric. It represents the **average absolute difference between our predicted sales and the actual sales**. So, on average, our model's predictions for `Item_Outlet_Sales` are off by approximately **$739.40**. This gives a straightforward understanding of the typical error magnitude in our forecasts.

**3. Overfitting/Underfitting Assessment: Is our model balanced?**

*   **Training R²: 0.721**
*   **Test R²: 0.590**

Comparing our model's performance on the data it was trained on (training R²) versus new, unseen data (test R²), we see a moderate difference. The training R-squared is 0.721, while the test R-squared is 0.590. This indicates that our model is **slightly overfitting**. This means it performs somewhat better on the data it has already seen than on new data. However, the drop isn't drastic, and our hyperparameter tuning efforts successfully reduced the severe overfitting we observed in the default Random Forest model. The current balance is reasonable, but further fine-tuning could potentially bridge this gap a bit more and improve generalization even further.